# GeoAI Project: Visualization for Progress Report
This notebook generates maps and charts for the Colombo North urban planning and flood risk analysis project.

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
import os
import json
from shapely.geometry import box

# Settings for high-quality plots
plt.rcParams['figure.dpi'] = 120
plt.style.use('ggplot')

# Paths
CACHE_DIR = '../backend/cache'
OSM_DATA = '../data/raw/osm_features_centroids.geojson'
STUDY_AREA_BBOX = [79.85, 6.92, 79.96, 6.99]

## 1. Map Visualization: Study Area and Flood Risk

In [ ]:
print("Loading spatial layers...")
buildings = gpd.read_file(os.path.join(CACHE_DIR, 'colombo_north_buildings.geojson'))
flood_zones = gpd.read_file(os.path.join(CACHE_DIR, 'flood_zones_clipped.geojson'))
roads = gpd.read_file(os.path.join(CACHE_DIR, 'processed_roads_clipped.geojson'))

fig, ax = plt.subplots(figsize=(8, 6))

# 1. Plot Roads (Base)
roads.plot(ax=ax, color='gray', linewidth=0.5, alpha=0.5, label='Road Network')

# 2. Plot Buildings
buildings.plot(ax=ax, color='blue', markersize=0.1, alpha=0.3, label='Building Footprints')

# 3. Plot Flood Risk Zones (Overlay)
flood_zones.plot(ax=ax, color='red', alpha=0.5, label='High Flood Risk Zone')

ax.set_title("Colombo North: Urban Infrastructure and Flood Risk Overlay", fontsize=12)
ax.set_xlabel("Longitude", fontsize=10)
ax.set_ylabel("Latitude", fontsize=10)
plt.legend(loc='upper right', fontsize=8)
plt.tight_layout()
plt.show()

## 2. Infrastructure Distribution Analysis (Non-Residential Pie Chart)

In [ ]:
print("Extracting infrastructure stats...")
osm_gdf = gpd.read_file(OSM_DATA)

# Define the boundary for clipping
boundary = box(*STUDY_AREA_BBOX)
boundary_gdf = gpd.GeoDataFrame({'geometry': [boundary]}, crs="EPSG:4326")
osm_clipped = gpd.clip(osm_gdf, boundary_gdf)

# Safe helper to count items even if column doesn't exist
def count_infra(df, col, values):
    if col not in df.columns:
        return 0
    return len(df[df[col].isin(values)])

# Map categories according to requested list
infra_counts = {
    "Schools": count_infra(osm_clipped, 'amenity', ['school', 'kindergarten', 'college', 'university']),
    "Religious Places": count_infra(osm_clipped, 'amenity', ['place_of_worship']),
    "Hospitals/Clinics": count_infra(osm_clipped, 'amenity', ['hospital', 'clinic']),
    "Parks": count_infra(osm_clipped, 'leisure', ['park', 'playground']),
    "Grounds/Fields": count_infra(osm_clipped, 'leisure', ['pitch']),
    "Police": count_infra(osm_clipped, 'amenity', ['police']),
    "Rail Station": count_infra(osm_clipped, 'railway', ['station'])
}

df_infra = pd.DataFrame(list(infra_counts.items()), columns=['Category', 'Count'])
df_infra = df_infra[df_infra['Count'] > 0]

if not df_infra.empty:
    plt.figure(figsize=(6, 6))
    colors = plt.cm.Paired(range(len(df_infra)))
    plt.pie(df_infra['Count'], labels=df_infra['Category'], 
            autopct='%1.1f%%', startangle=140, colors=colors, 
            shadow=True, pctdistance=0.80, textprops={'fontsize': 8})
    centre_circle = plt.Circle((0,0),0.70,fc='white')
    plt.gcf().gca().add_artist(centre_circle)
    plt.title("Distribution of Key Public Infrastructure", fontsize=12)
    plt.axis('equal') 
    plt.show()

## 3. High-Density Building Hotspots (Heatmap)

In [ ]:
print("Generating building density heatmap...")

# Calculate centroids for density analysis
buildings['centroid_x'] = buildings.geometry.centroid.x
buildings['centroid_y'] = buildings.geometry.centroid.y

fig, ax = plt.subplots(figsize=(8, 7))

# Plot the building density using a hexbin plot
hb = ax.hexbin(buildings['centroid_x'], buildings['centroid_y'], gridsize=50, cmap='YlOrRd', mincnt=1, alpha=0.9)

# Add roads as an overlay in light color for reference
roads.plot(ax=ax, color='black', alpha=0.1, linewidth=0.3)

cb = fig.colorbar(hb, ax=ax)
cb.set_label('Building Count (Density)', fontsize=10)

ax.set_title("Building Density Hotspots: Colombo North & Expansion Area", fontsize=12)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.tight_layout()
plt.show()

print("Heatmap generated. Darker red areas indicate the highest concentrations of urban infrastructure.")

## 4. Key Findings Summary
- **Urban Density**: The hexbin heatmap confirms that building density is highest in the core Modara/Grandpass wards, tapering off slightly towards the Kelaniya expansion.
- **Spatial Risk**: Comparing the density map with the flood overlay (Map 1) shows that the highest density zones are directly congruent with historical high-risk flood inundation regions.
- **Infrastructure Gaps**: Public facilities represent only a small fraction of the total urban landscape, necessitating highly prioritized resource allocation during risk mitigation.